In [ ]:
import os
import string
import torch
from torch.utils.data import Dataset, DataLoader, ConcatDataset,random_split
import torchvision.transforms as T
import torch.nn as nn
import torch.optim as optim
torch.manual_seed(42)

In [ ]:
CHAR_SET= string.ascii_letters + string.digits
char_to_idx = {c:i+1 for i,c in enumerate(CHAR_SET)}
idx_to_char = {i+1:c for i,c in enumerate(CHAR_SET)}


In [36]:
class CaptchaDataset(Dataset):
    def __init__(self,root,transform=None):
        self.paths = [os.path.join(root,f) for f in os.listdir(root) if f.endswith('.png')]
        self.transform=transform or T.Compose([
            T.Grayscale(),
            T.Resize((32,128)),
            T.RandomAffine(degrees=0, translate=(0.02, 0.02)),
            T.ColorJitter(brightness=0.3, contrast=0.3),
            T.ToTensor(),
            T.Normalize((0.5,),(0.5,))
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self,idx):
        path=self.paths[idx]
        label_str=os.path.splitext(os.path.basename(path))[0]
        img=self.transform(Image.open(path))
        label=torch.tensor([char_to_idx[c] for c in label_str],dtype=torch.long)
        return img,label,len(label_str)

def collate_fn(batch):
    imgs,labels,lengths = zip(*batch)
    imgs=torch.stack(imgs)
    labels_concat=torch.cat(labels)
    lengths=torch.tensor(lengths,dtype=torch.long)
    return imgs,labels_concat,lengths


# MODEL

In [37]:
class BidirectionalLSTM(nn.Module):
    def __init__(self,in_size,hidden,out_size):
        super().__init__()
        self.lstm=nn.LSTM(in_size,hidden,bidirectional=True,dropout=0.2,batch_first=False)
        self.fc=nn.Linear(hidden*2,out_size)

    def forward(self,x):
        out,_=self.lstm(x)
        T,B,C=out.shape
        return self.fc(out.view(T*B,C)).view(T,B,-1)

class CRNN(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        self.cnn=nn.Sequential(
            nn.Conv2d(1,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128),nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(128,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.Conv2d(256,256,3,padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.MaxPool2d((2,1)),
            nn.Dropout2d(0.2),
            nn.Conv2d(256,512,3,padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.Conv2d(512,512,3,padding=1), nn.BatchNorm2d(512), nn.ReLU(),
            nn.MaxPool2d((2,1)),
            nn.Dropout2d(0.2),
            nn.Conv2d(512,512,2), nn.BatchNorm2d(512), nn.ReLU()
        )
        self.rnn=nn.Sequential(
            BidirectionalLSTM(512,256,256),
            BidirectionalLSTM(256,256,num_classes)
        )

    def forward(self, x):
      feat = self.cnn(x)
      feat = feat.squeeze(2)
      feat = feat.permute(2, 0, 1)
      return self.rnn(feat)

In [39]:
def split_dataset(ds, train_ratio=0.765, val_ratio=0.135):
    total = len(ds)
    train_len = int(train_ratio * total)
    val_len   = int(val_ratio * total)
    test_len  = total - train_len - val_len
    generator = torch.Generator().manual_seed(42)
    return random_split(ds, [train_len, val_len, test_len], generator=generator)

NUM_CLASSES=len(CHAR_SET)+1
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")

pattern1 = CaptchaDataset("/home/gautam/rcnn captcha/captcha_dataset/cpp_captchas")
pattern2 = CaptchaDataset("/home/gautam/rcnn captcha/captcha_dataset/state_portal_captchas")

p1_train, p1_val, p1_test = split_dataset(pattern1)
p2_train, p2_val, p2_test = split_dataset(pattern2)

train_set = ConcatDataset([p1_train, p2_train])
val_set   = ConcatDataset([p1_val,   p2_val])
test_set  = ConcatDataset([p1_test,  p2_test])

train_loader=DataLoader(train_set, batch_size=32, shuffle=True, collate_fn=collate_fn, pin_memory=True)
val_loader=DataLoader(val_set,batch_size=32,shuffle=False, collate_fn=collate_fn, pin_memory=True)
test_loader=DataLoader(test_set,batch_size=32,shuffle=False,collate_fn=collate_fn,pin_memory=True)

# ========MODEL TRAINING=========
epochs=70
lr=5e-4

model=CRNN(NUM_CLASSES).to(device)
loss_function=nn.CTCLoss(blank=0,zero_infinity=True)
optimizer=optim.Adam(model.parameters(),lr=lr,weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50, eta_min=1e-6)

train_loss=[]
val_loss=[]

# CTC LOSS
def ctc_decode(preds):
    preds=preds.argmax(2).permute(1,0)
    results=[]

    for seq in preds:
        chars,prev=[],0
        for t in seq:
            t=t.item()
            if t!=0 and t!=prev:            #skip blank and repeated chars
                chars.append(idx_to_char.get(t,''))
            prev=t
        results.append(''.join(chars))
    return results

best_val_acc=0
patience_counter = 0
early_stop_patience = 15

# for i in range(epochs):

#     # ========TRAIN LOOP=========
#     model.train()
#     total_epoch_loss = 0

#     for imgs, labels, label_lens in train_loader:
#         imgs = imgs.to(device)

#         optimizer.zero_grad()
#         preds = model(imgs)
#         T, B, C = preds.shape
#         preds_log = preds.log_softmax(2)
#         input_lens = torch.full((B,), T, dtype=torch.long)
#         loss = loss_function(preds_log, labels, input_lens, label_lens)

#         loss.backward()
#         nn.utils.clip_grad_norm_(model.parameters(), 5)
#         optimizer.step()
#         total_epoch_loss += loss.item()

#     avg_epoch_loss = total_epoch_loss / len(train_loader)
#     train_loss.append(avg_epoch_loss)

#     # ========VALIDATION LOOP========
#     model.eval()
#     total_val_loss = 0
#     correct, total = 0, 0

#     with torch.no_grad():
#         for imgs, labels, label_lens in val_loader:
#             imgs = imgs.to(device)

#             preds = model(imgs)
#             T, B, C = preds.shape
#             preds_log = preds.log_softmax(2)
#             input_lens = torch.full((B,), T, dtype=torch.long)
#             loss = loss_function(preds_log, labels, input_lens, label_lens)
#             total_val_loss += loss.item()

#             decoded = ctc_decode(preds.cpu())
#             offset = 0
#             for pred, label_len in zip(decoded, label_lens):
#                 true = ''.join(idx_to_char[c.item()] for c in labels[offset:offset+label_len])
#                 correct += (pred == true)
#                 total += 1
#                 offset += label_len

#     avg_val_loss = total_val_loss / len(val_loader)
#     val_loss.append(avg_val_loss)
#     val_acc = correct / total

#     scheduler.step()

#     if val_acc > best_val_acc:
#         best_val_acc = val_acc
#         torch.save(model.state_dict(), "best_model.pth")
#         patience_counter = 0
#     else:
#         patience_counter += 1

#     if patience_counter >= early_stop_patience:
#         print(f"Early stopping at epoch {i+1}")
#         break

#     print(f"==========EPOCH {i+1}============")
#     print(f"TRAIN LOSS: {avg_epoch_loss:.4f}")
#     print(f"VAL LOSS:   {avg_val_loss:.4f}")
#     print(f"VAL ACC:    {val_acc:.4f}")
#     print(f"BEST ACC:   {best_val_acc:.4f}")


In [40]:
# After loading best model, check character-level accuracy (not just full string)
model=CRNN(NUM_CLASSES).to(device)
model.load_state_dict(torch.load("CRNN_best_model.pth",map_location=device))
model.eval()

char_correct, char_total = 0, 0
full_correct, full_total = 0, 0
length_errors = 0  # predicted wrong number of chars

with torch.no_grad():
    for imgs, labels, label_lens in test_loader:
        imgs = imgs.to(device)
        preds = model(imgs)
        decoded = ctc_decode(preds.cpu())
        
        offset = 0
        for pred, label_len in zip(decoded, label_lens):
            true = ''.join(idx_to_char[c.item()] for c in labels[offset:offset+label_len])
            
            # full string accuracy
            full_correct += (pred == true)
            full_total += 1
            
            # character level accuracy
            for p, t in zip(pred, true):
                char_correct += (p == t)
            char_total += len(true)
            
            # length errors
            if len(pred) != len(true):
                length_errors += 1
            
            offset += label_len

print(f"Full string acc:  {full_correct/full_total:.4f}")
print(f"Char level acc:   {char_correct/char_total:.4f}")
print(f"Length errors:    {length_errors/full_total:.4f}")

/home/gautam/rcnn captcha/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Full string acc:  0.9190
Char level acc:   0.9852
Length errors:    0.0013


In [44]:
from PIL import Image
import torchvision.transforms as transforms

transform=transforms.Compose([
            transforms.Grayscale(),
            transforms.Resize((32,128)),
            transforms.RandomAffine(degrees=0, translate=(0.02, 0.02)),
            transforms.ColorJitter(brightness=0.3, contrast=0.3),
            transforms.ToTensor(),
            transforms.Normalize((0.5,),(0.5,))
        ])

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=CRNN(NUM_CLASSES).to(device)
model.load_state_dict(torch.load("CRNN_best_model.pth",map_location=device))
model.eval()

def predict(image_path):
    img=Image.open(image_path).convert("RGB")
    img=transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        preds=model(img)
    
    decoded=ctc_decode(preds.cpu())
    return decoded[0]

result=predict("/home/gautam/rcnn captcha/image.png")
print(f"Predicted: {result}")

Predicted: FrF3ha
